### Libraries

In [ ]:
!pip install -q \
  datasets \
  transformers \
  peft \
  huggingface_hub \
  ipywidgets

In [9]:
import os
import torch
import ipywidgets
from torch.utils.data import Dataset
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from itertools import islice
from huggingface_hub import login
from huggingface_hub import HfFolder
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

### Load model & dataset

##### Login to huggingface (only in Colab needed)

In [10]:
login()

##### Load base model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model     = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype="auto")

##### Load fine-tuned model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("eduhuemar001/tinyllama-german")
model     = AutoModelForCausalLM.from_pretrained("eduhuemar001/tinyllama-german")

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

##### Load dataset

In [ ]:
streamed_dataset = load_dataset(
    "wikipedia",
    "20220301.de",
    split="train",
    streaming=True,
    trust_remote_code=True
)

dataset = list(islice(streamed_dataset, 1000)) # Donwload 1000 wikipedia articles

### Preprocessing

##### Tokenize data

In [ ]:
tokens = []

for item in dataset:
    ids = tokenizer(item["text"], return_attention_mask=False, add_special_tokens=False).input_ids
    tokens.extend(ids + [tokenizer.eos_token_id])  # optional: add EOS after each article

##### Split tokens into chunks

In [ ]:
block_size = 512
total_length = len(tokens) - (len(tokens) % block_size)
tokens = tokens[:total_length] # Ensure the tokens array has a length of multiple of block size

chunks = [tokens[i:i + block_size] for i in range(0, total_length, block_size)] # Split tokens into 512-token chunks

##### Prepare chunks for training

In [ ]:
class ChunkDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        ids = torch.tensor(self.chunks[idx], dtype=torch.long)
        return {
            "input_ids": ids,
            "labels": ids
        }

train_dataset = ChunkDataset(chunks)

### Set training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama-german-finetuned", # Where to save checkpoints
    per_device_train_batch_size=2, # 2 chunks are getting processed by GPU at the same time
    gradient_accumulation_steps=4, # Weights are being updated after 8 chunks are processed (2 chunks * 4 batches = 8 chunks)
    num_train_epochs=2,            # Model uses each chunk 2 times for training (dataset is processed 2 times iteratively)
    learning_rate=2e-4,            # Speed of learning (gradient step size)
    save_strategy="epoch",         # Save a model checkpoint after each epoch
    logging_steps=10,              # Print loss/log info every 10 steps
    fp16=True,                     # Use mixed precision (faster/lower memory)
    report_to="none",              # No external logging
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

### Start training loop

In [ ]:
trainer.train()

### Save data

##### Save to hugginface

In [ ]:
model.push_to_hub("eduhuemar001/tinyllama-german")
tokenizer.push_to_hub("eduhuemar001/tinyllama-german")

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/eduhuemar001/tinyllama-german/commit/e5a0e6e772bb9d9202b55f0a638cbd5cf7b9326d', commit_message='Upload tokenizer', commit_description='', oid='e5a0e6e772bb9d9202b55f0a638cbd5cf7b9326d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/eduhuemar001/tinyllama-german', endpoint='https://huggingface.co', repo_type='model', repo_id='eduhuemar001/tinyllama-german'), pr_revision=None, pr_num=None)